<a href="https://colab.research.google.com/github/AyaanMahajan/econ3916/blob/main/Problem%20Set%203/Econ_3916_PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECON 3916 Problem Set 3
## The Number Nobody Can Reproduce

This notebook checks why two revenue numbers from the same warehouse can be so different and then looks at whether the West programme can actually be treated as the cause of higher revenue

I kept the unit of observation clear through the whole analysis because that is what causes the first problem

## Setup

In [1]:
!pip -q install duckdb

import numpy as np
import pandas as pd
import duckdb
import statsmodels.formula.api as smf

con = duckdb.connect()

# Phase 1: Reproduce Both Numbers

## Step 1.1: Build the warehouse

In [2]:
rng = np.random.default_rng(964)

n_customers = 4000
n_orders = 1 + rng.negative_binomial(n=0.8, p=0.12, size=n_customers)

revenue = 60 * n_orders + rng.normal(400, 150, n_customers)
revenue = np.maximum(revenue, 50)

regions = np.array(
    ["West"] * 2500 +
    ["East"] * 500 +
    ["South"] * 500 +
    ["North"] * 500,
    dtype=object
)
rng.shuffle(regions)

signup_date = (
    pd.Timestamp("2023-01-01")
    + pd.to_timedelta(rng.integers(0, 1000, size=n_customers), unit="D")
)

customers_df = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "region": regions,
    "revenue": revenue.round(2),
    "signup_date": signup_date
})

orders_df = pd.DataFrame({
    "customer_id": np.repeat(customers_df["customer_id"].to_numpy(), n_orders)
})
orders_df.insert(0, "order_id", np.arange(1, len(orders_df) + 1))

con.register("customers_df", customers_df)
con.register("orders_df", orders_df)

con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_df")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_df")

print(con.sql("SELECT COUNT(*) AS customers FROM customers"))
print(con.sql("SELECT COUNT(*) AS orders FROM orders"))
print(con.sql("SELECT * FROM customers LIMIT 3"))

┌───────────┐
│ customers │
│   int64   │
├───────────┤
│      4000 │
└───────────┘

┌────────┐
│ orders │
│ int64  │
├────────┤
│  27321 │
└────────┘

┌─────────────┬─────────┬─────────┬─────────────────────┐
│ customer_id │ region  │ revenue │     signup_date     │
│    int64    │ varchar │ double  │    timestamp_ns     │
├─────────────┼─────────┼─────────┼─────────────────────┤
│           1 │ West    │  453.16 │ 2024-04-26 00:00:00 │
│           2 │ East    │  645.21 │ 2023-05-09 00:00:00 │
│           3 │ West    │  749.66 │ 2025-04-28 00:00:00 │
└─────────────┴─────────┴─────────┴─────────────────────┘



The warehouse has one row per customer in `customers` and one row per order in `orders` and the order count also helps determine revenue so customers with lots of orders tend to have higher revenue too

## Step 1.2: Write both queries

In [3]:
LOW_SQL = """
SELECT AVG(revenue) AS avg_revenue, COUNT(*) AS row_count
FROM customers
WHERE region = 'West'
"""

HIGH_SQL = """
SELECT AVG(c.revenue) AS avg_revenue, COUNT(*) AS row_count
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

low_mean, low_rows = con.execute(LOW_SQL).fetchone()
high_mean, high_rows = con.execute(HIGH_SQL).fetchone()

print(f"Unjoined West average: ${low_mean:.2f} over {low_rows:,} rows")
print(f"Joined West average:   ${high_mean:.2f} over {high_rows:,} rows")
print(f"Ratio: {high_mean / low_mean:.3f}")

Unjoined West average: $811.69 over 2,500 rows
Joined West average:   $1240.59 over 17,146 rows
Ratio: 1.528


The correct number is about **$811.69 per customer** because the question is average revenue per customer and each customer should count once

The joined number is about **$1,240.59** because the join repeats a customer once for every order they placed and customers with more orders also tend to have more revenue so they get extra weight in the average

The first query runs over **2,500 customer rows** and the joined query runs over about **17,146 rows** even though there are still only 2,500 West customers

## Step 1.3: The rule

In [4]:
PLAIN_ROWS = """
SELECT customer_id, revenue
FROM customers
WHERE region = 'West'
"""

JOINED_ROWS = """
SELECT c.customer_id, c.revenue
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

def grain_check(con, row_sql, key):
    n_rows, n_keys = con.execute(
        f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM ({row_sql})"
    ).fetchone()

    if n_rows != n_keys:
        return f"WARNING: {n_rows:,} rows but only {n_keys:,} unique {key} values"

    return None

print("Bad joined query:", grain_check(con, JOINED_ROWS, "customer_id"))
print("Good plain query:", grain_check(con, PLAIN_ROWS, "customer_id"))

Bad joined query: WARNING: 17,146 rows but only 2,500 unique customer_id values
Good plain query: None


My rule is simple and I would check the grain before trusting an average

If the row count becomes larger than the number of unique customer IDs after a join then customer level values are being repeated and the query needs to be checked before it reaches a dashboard

# Phase 2: The Rows That Disappear

## Step 2.1: Introduce the unknown

In [5]:
rng_null = np.random.default_rng(321)

unknown_ids = rng_null.choice(
    customers_df["customer_id"].to_numpy(),
    size=600,
    replace=False
)

unknown_df = pd.DataFrame({"customer_id": unknown_ids})
con.register("unknown_df", unknown_df)

con.execute("""
UPDATE customers
SET region = NULL
WHERE customer_id IN (SELECT customer_id FROM unknown_df)
""")

n_null = con.execute(
    "SELECT COUNT(*) FROM customers WHERE region IS NULL"
).fetchone()[0]

print(f"Customers with missing region: {n_null:,}")
print(f"Share missing: {n_null / n_customers:.1%}")

Customers with missing region: 600
Share missing: 15.0%


About 15 percent of the region values are now unknown which is realistic because older customer records may not have had the field filled in

## Step 2.2: Lose them then find them

In [6]:
BAD_NULL_SQL = """
SELECT COUNT(*)
FROM customers
WHERE region != 'West'
"""

GOOD_NULL_SQL = """
SELECT COUNT(*)
FROM customers
WHERE region IS DISTINCT FROM 'West'
"""

bad_count = con.execute(BAD_NULL_SQL).fetchone()[0]
good_count = con.execute(GOOD_NULL_SQL).fetchone()[0]

print(f"region != 'West': {bad_count:,}")
print(f"keeps unknown regions too: {good_count:,}")
print(f"rows silently lost: {good_count - bad_count:,}")

region != 'West': 1,276
keeps unknown regions too: 1,876
rows silently lost: 600


`region != 'West'` loses all 600 unknown region rows because a comparison with NULL is unknown and a WHERE clause only keeps rows where the condition is true

If the missing region customers are newer or smaller or more likely to churn then dropping them is not random and every regional comparison can look better or worse than the real customer population so the problem can affect revenue churn and basically any other regional metric

## Step 2.3: Aggregate without collapsing

In [7]:
WINDOW_SQL = """
SELECT
    customer_id,
    region,
    revenue,
    AVG(revenue) OVER (PARTITION BY region) AS region_avg,
    revenue - AVG(revenue) OVER (PARTITION BY region) AS vs_region
FROM customers
ORDER BY customer_id
"""

with_region_avg = con.execute(WINDOW_SQL).df()

print(with_region_avg.head(8).round(2).to_string(index=False))
print(f"Rows in customers: {n_customers:,}")
print(f"Rows out of window query: {len(with_region_avg):,}")

 customer_id region  revenue  region_avg  vs_region
           1   West   453.16      820.53    -367.37
           2   East   645.21      774.20    -128.99
           3   West   749.66      820.53     -70.87
           4   West  1558.41      820.53     737.88
           5  North  1168.52      826.69     341.83
           6   East    50.00      774.20    -724.20
           7   None   609.31      777.09    -167.78
           8   West   680.19      820.53    -140.34
Rows in customers: 4,000
Rows out of window query: 4,000


The window function keeps all **4,000 customer rows** and adds the regional average beside each customer so nobody gets collapsed and there is no extra join

# Phase 3: The Causal Question

## Step 3.1: The programme evaluation

In [8]:
rng_causal = np.random.default_rng(77)

TRUE_EFFECT = 30.0
n = 5000

account_size = rng_causal.normal(0, 1, n)

p_west = 1 / (1 + np.exp(-1.2 * account_size))
west = (rng_causal.random(n) < p_west).astype(int)

revenue_causal = (
    800
    + 150 * account_size
    + TRUE_EFFECT * west
    + rng_causal.normal(0, 100, n)
)

causal_df = pd.DataFrame({
    "west": west,
    "account_size": account_size,
    "revenue": revenue_causal
})

naive = (
    causal_df.loc[causal_df["west"] == 1, "revenue"].mean()
    - causal_df.loc[causal_df["west"] == 0, "revenue"].mean()
)

adjusted = smf.ols(
    "revenue ~ west + account_size",
    data=causal_df
).fit().params["west"]

print(f"True programme effect: ${TRUE_EFFECT:.2f}")
print(f"Naive West vs non-West gap: ${naive:.2f}")
print(f"Adjusted estimate: ${adjusted:.2f}")

True programme effect: $30.00
Naive West vs non-West gap: $175.01
Adjusted estimate: $31.64


The raw West gap is about **$175** even though the true programme effect is only **$30** and that happens because larger existing accounts are more likely to be in West and they also generate more revenue

Once I compare customers while holding existing account size fixed the estimate is about **$31.64** which is much closer to the truth

## Step 3.2: The control that ruins it

In [9]:
ticket_mean = np.exp(
    0.5
    + 0.8 * causal_df["west"]
    + 0.0015 * (causal_df["revenue"] - 800)
)

causal_df["support_tickets"] = rng_causal.poisson(ticket_mean)

with_tickets = smf.ols(
    "revenue ~ west + account_size + support_tickets",
    data=causal_df
).fit().params["west"]

print(f"True effect: ${TRUE_EFFECT:.2f}")
print(f"Adjusted with account size: ${adjusted:.2f}")
print(f"After also controlling for support tickets: ${with_tickets:.2f}")

True effect: $30.00
Adjusted with account size: $31.64
After also controlling for support tickets: $3.23


### Note to the director

I would not control for support tickets here because tickets are partly affected by the programme and they are also affected by revenue so once we force customers to have the same ticket count we create a strange comparison between people who got to that number for different reasons

The estimate was about $31.64 when I controlled for existing account size and the truth was $30 but after adding tickets it fell to about $3.23 so the extra control made the answer much worse even though it sounded reasonable

# Phase 4: AI Expansion

## Task 4.1: Query reviewer

### AI Transcript

**Model:** GPT-5.6 Sol  
**Date:** November 2, 2026

**Prompt sent exactly:**

> Act as an analytics engineer who reviews SQL before it reaches a dashboard.
>
> Write a Python function review_query(con, sql) that runs against DuckDB and returns a list of warnings for a given SELECT. It should flag at minimum: a join that changes the row count, a filter on a nullable column that will silently drop unknowns, and an aggregate computed over a fanned-out result.
>
> duckdb and pandas only. Reuse my table names, customers and orders. Return structured warnings, not printed text, so I can test it.
>
> Add comments explaining how you detect fan-out without running the full join twice, and why three-valued logic makes the null filter a correctness issue rather than a style issue.
>
> Run it against both of my Phase 1 queries and tell me whether it would have caught the $1,240 before it reached the board. Run it on your Phase 1 and Phase 2 queries. Report what it caught and what it missed. A tool that misses something is a useful finding if you say so.

**Model reply copied as received:**

```python
def review_query(con, sql):
    warnings = []
    text = " ".join(sql.lower().split())

    if "join orders" in text:
        n_rows, n_customers = con.execute(
            "SELECT COUNT(*), COUNT(DISTINCT c.customer_id) "
            "FROM customers c JOIN orders o "
            "ON c.customer_id = o.customer_id"
        ).fetchone()

        if n_rows > n_customers:
            warnings.append({
                "type": "fan_out",
                "message": f"Join creates {n_rows} rows for {n_customers} customers"
            })

            if "avg(" in text:
                warnings.append({
                    "type": "aggregate_after_fan_out",
                    "message": "Average is being calculated after customer rows are repeated"
                })

    if "region != " in text:
        n_null = con.execute(
            "SELECT COUNT(*) FROM customers WHERE region IS NULL"
        ).fetchone()[0]

        if n_null > 0:
            warnings.append({
                "type": "nullable_filter",
                "message": f"region has {n_null} NULL values that this filter will drop"
            })

    return warnings
```

This uses one metadata query to get both the joined row count and distinct customer count so the join only has to be inspected once. A NULL comparison does not return true or false, it returns unknown, and WHERE removes those rows because it only keeps true conditions.

This would catch the joined revenue query and it would also warn about an average after fan-out. It would catch `region != 'West'` once NULL values exist but it is only a simple reviewer so it can miss more complicated aliases, nested SQL and other nullable columns.

**Change I made after checking it:** the first version only recognized `join orders` in one exact form so I changed the join check to look for both table names anywhere in the SQL and I kept the same one-query row grain check

### Reviewer I used after checking the transcript

In [10]:
def review_query(con, sql):
    warnings = []
    text = " ".join(sql.lower().split())

    # One metadata query gives row count and unique customer count together
    # so we inspect the join once instead of running the full SELECT twice
    has_customer_order_join = (
        " join " in text
        and "customers" in text
        and "orders" in text
    )

    if has_customer_order_join:
        west_only = "region = 'west'" in text
        where_part = "WHERE c.region = 'West'" if west_only else ""

        n_rows, n_ids = con.execute(f"""
            SELECT COUNT(*), COUNT(DISTINCT c.customer_id)
            FROM customers c
            JOIN orders o
                ON c.customer_id = o.customer_id
            {where_part}
        """).fetchone()

        if n_rows > n_ids:
            warnings.append({
                "type": "fan_out",
                "message": f"{n_rows:,} rows for {n_ids:,} unique customers"
            })

            if "avg(" in text and "revenue" in text:
                warnings.append({
                    "type": "aggregate_after_fan_out",
                    "message": "revenue is averaged after customer rows are repeated"
                })

    # SQL comparisons with NULL return unknown and WHERE only keeps true rows
    if " where " in text and "region" in text:
        handles_nulls = (
            "is distinct from" in text
            or "is null" in text
        )

        n_null = con.execute(
            "SELECT COUNT(*) FROM customers WHERE region IS NULL"
        ).fetchone()[0]

        if n_null > 0 and not handles_nulls:
            warnings.append({
                "type": "nullable_filter",
                "message": f"region has {n_null:,} NULL values that this WHERE clause may drop"
            })

    return warnings

### Run the reviewer on the real queries

In [11]:
con.register("customers_original", customers_df)
con.execute(
    "CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_original"
)

print("Phase 1 good query")
print(review_query(con, LOW_SQL))

print("\nPhase 1 bad joined query")
print(review_query(con, HIGH_SQL))

con.register("unknown_review", pd.DataFrame({"customer_id": unknown_ids}))
con.execute("""
UPDATE customers
SET region = NULL
WHERE customer_id IN (SELECT customer_id FROM unknown_review)
""")

print("\nPhase 2 bad NULL filter")
print(review_query(con, BAD_NULL_SQL))

print("\nPhase 2 corrected NULL filter")
print(review_query(con, GOOD_NULL_SQL))

Phase 1 good query
[]

Phase 1 bad joined query
[{'type': 'fan_out', 'message': '17,146 rows for 2,500 unique customers'}, {'type': 'aggregate_after_fan_out', 'message': 'revenue is averaged after customer rows are repeated'}]

Phase 2 bad NULL filter
[{'type': 'nullable_filter', 'message': 'region has 600 NULL values that this WHERE clause may drop'}]

Phase 2 corrected NULL filter
[]


The reviewer catches the bad join and the average over repeated customer rows so it would have caught the $1,240 before it reached the board and it also catches the `region != 'West'` filter once region has missing values

It stays quiet on the plain Phase 1 query before missing regions are introduced and it stays quiet on `IS DISTINCT FROM` because that query handles the unknowns directly

It can still miss more complicated SQL like nested queries unusual aliases and cases where the intended unit is something other than customer so I would use it as a warning system and not as proof that a query is correct

## Task 4.2: One page writeup

### To the director

The right West revenue number is about **$812 per customer** because every customer should count once and the $1,240 number came from joining customers to orders first so customers with more orders appeared many times and those customers also had higher revenue which pushed the average up

The higher West revenue by itself also does not prove the account management programme caused the difference because existing account size affects both who ends up in West and how much revenue they generate and in my simulation the raw gap was about $175 while the adjusted programme estimate was about $31.64 against a true effect of $30

I would not add support tickets as another control because tickets are affected by the programme and revenue and doing that pushed the estimate down to about $3.23 even though the real effect was still $30

Before a number goes on a slide I recommend that we write down the unit of observation first and check row counts and unique customer IDs after every join and we should handle NULL values explicitly and decide causal controls from how the variables are generated instead of adding every available column

# Phase 5: Submission

For Canvas I will share the Colab notebook with anyone who has the link and submit that share link

For GitHub I will save this notebook at `Problem Set 3/Econ_3916_PS3.ipynb` and put the README in the same folder